# `aidp-fusion-bundle` orchestrator — inline run

Three-cell demonstration of the **architectural primary** execution path: the orchestrator runs in-process inside an AIDP notebook session with access to the AIDP-injected `SparkSession`, `checkpointer`, `aidputils.secrets`, and Delta catalog. None of those exist outside this surface — see `PLAN_P1.5_orchestrator.md` §4.5 "Execution surfaces" for why `--inline` is the primary, not a debug fallback.

**Pre-requisites:** customer has copied the plugin checkout into the AIDP workspace, placed their `bundle.yaml` next to this notebook, and exported the BICC credential reference (`${vault:OCID}` recommended; `${env:VAR}` for dev iteration).

**Run order:** seed mode rebuilds bronze → silver → gold from BICC every invocation. After the first successful seed run, the state-table query at the bottom shows one row per dataset with `last_run_at`, `status`, `row_count`, `silver_run_id` / `gold_run_id`, and `skip_reason`.


In [ ]:
# Cell 1 — bundle path + smoke-import the orchestrator package
from pathlib import Path

from oracle_ai_data_platform_fusion_bundle import orchestrator

BUNDLE_PATH = Path("bundle.yaml")  # adjust if your bundle lives elsewhere

assert BUNDLE_PATH.exists(), f"bundle not found: {BUNDLE_PATH.resolve()}"
print(f"orchestrator package loaded; bundle at {BUNDLE_PATH.resolve()}")
print(f"public surface: {[n for n in orchestrator.__all__ if not n.startswith('_')]}")


In [ ]:
# Cell 2 — seed run
# Passes the AIDP-injected `spark` so the orchestrator skips its
# `_bootstrap_spark()` fallback (which constructs a fresh session).
# Inside an AIDP notebook session, `spark` is a global.
summary = orchestrator.run(
    bundle_path=BUNDLE_PATH,
    spark=spark,            # noqa: F821 — AIDP-injected global
    mode="seed",
    datasets=None,           # None = every dataset/dim/mart in bundle.yaml
    layers=None,             # None = bronze + silver + gold
    dry_run=False,
)

# Print one line per step — matches the CLI's table without Rich
print(f"run_id={summary.run_id}")
print(f"steps: {summary.succeeded} ok, {summary.failed} failed, "
      f"{summary.skipped} skipped, {summary.deferred} deferred "
      f"({summary.total_duration_seconds:.1f}s total)")
for step in summary.steps:
    skip_tag = f" [{step.skip_reason}]" if step.skip_reason else ""
    rc = step.row_count if step.row_count is not None else "-"
    print(f"  {step.layer:6s}  {step.dataset_id:24s}  {step.status:10s}{skip_tag:12s}  rows={rc}")

# Surface the run for the AIDP MCP / REST-job-output channel — both
# the stdout marker pattern (proven in the workbench-spark-connectors
# live_test_driver) and oidlUtils.notebook.exit() if available.
import json
_result_payload = {
    "run_id": summary.run_id,
    "bundle_project": summary.bundle_project,
    "mode": summary.mode,
    "succeeded": summary.succeeded,
    "failed": summary.failed,
    "skipped": summary.skipped,
    "deferred": summary.deferred,
    "total_duration_seconds": summary.total_duration_seconds,
}
print("AIDP_LIVE_TEST_RESULT_BEGIN", json.dumps(_result_payload), "AIDP_LIVE_TEST_RESULT_END")
try:
    from oidlUtils import notebook as _oidl_notebook  # type: ignore[import-not-found]
    _oidl_notebook.exit(json.dumps(_result_payload))
except Exception:
    # oidlUtils.notebook.exit() is AIDP-runtime-only; harmless to skip
    # in other surfaces (the stdout marker above is the reliable channel).
    pass


In [ ]:
# Cell 3 — inspect fusion_bundle_state for the run we just executed
# (verifies B3 SOX-trail audit columns + B1.1 skip_reason discriminator)
from oracle_ai_data_platform_fusion_bundle.config.paths import TablePaths
from oracle_ai_data_platform_fusion_bundle.orchestrator.runtime import load_bundle

_bundle, _paths = load_bundle(BUNDLE_PATH)
_state_table = _paths.bronze("fusion_bundle_state")

# Latest row per dataset (the same query commands/run.py::status() uses)
spark.sql(f"""
    WITH ranked AS (
      SELECT
        dataset_id, layer, mode, last_run_at, status,
        row_count, skip_reason, duration_seconds, run_id,
        ROW_NUMBER() OVER (PARTITION BY dataset_id
                           ORDER BY last_run_at DESC) AS rn
      FROM {_state_table}
      WHERE run_id = '{summary.run_id}'
    )
    SELECT dataset_id, layer, mode, status, row_count,
           skip_reason, duration_seconds, last_run_at
    FROM ranked
    WHERE rn = 1
    ORDER BY layer, dataset_id
""").show(truncate=False)

# Verify the SOX-trail join works on silver/gold — every produced
# row has silver_run_id / gold_run_id equal to summary.run_id.
for layer in ("silver", "gold"):
    rc_col = f"{layer}_run_id"
    print(f"\n=== sample {layer} rows (verify {rc_col} matches run_id) ===")
    # Pick the first non-deferred step in this layer
    candidate = next(
        (s for s in summary.steps
         if s.layer == layer and s.status == "success"),
        None,
    )
    if candidate is None:
        print(f"  (no successful {layer} rows in this run)")
        continue
    table = _paths.silver(candidate.dataset_id) if layer == "silver" else _paths.gold(candidate.dataset_id)
    spark.sql(f"SELECT {rc_col} FROM {table} LIMIT 3").show(truncate=False)
